In [ ]:
# 1. Instalação das Dependências

# Instala todas as bibliotecas necessárias
!pip install openai openai-whisper gtts pydub ipywidgets sounddevice numpy scipy

# Para detecção de silêncio (VAD)
!pip install webrtcvad

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 10.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.1 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=4aea3801e48f4299e2865045926425fbcfd90638a4070c04a535e45ba7043b36
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
# 2. Configurações e Importações

import whisper
from gtts import gTTS
from pydub import AudioSegment
from pydub.playback import play
from pydub.effects import normalize
import openai
import io
import IPython.display as ipd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Audio, clear_output
import base64
import time
import hashlib
import json
import os
from datetime import datetime
import threading
import queue

# Configuração da API OpenAI
openai.api_key = ""SUA_CHAVE_DE_API_AQUI"
# Configurações do sistema
CONFIG = {
    "modelo_whisper": "base",  # Opções: tiny, base, small, medium, large
    "idioma": "pt",
    "voz_gtts": "pt-br",  # Opções: pt-br, pt, en, etc.
    "velocidade_gtts": False,  # False = normal, True = lento
    "cache_habilitado": True,
    "vad_habilitado": True,
    "tempo_maximo_gravacao": 10,  # segundos
    "silencio_maximo": 1.5,  # segundos para considerar fim da fala
    "nivel_ruido": 500  # threshold para detecção de áudio
}

# Inicializações
modelo_whisper = whisper.load_model(CONFIG["modelo_whisper"])
print(f"✅ Modelo Whisper ({CONFIG['modelo_whisper']}) carregado!")

# Cria diretório para cache
if CONFIG["cache_habilitado"]:
    os.makedirs("cache", exist_ok=True)
    os.makedirs("historico", exist_ok=True)

100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 60.0MiB/s]


✅ Modelo Whisper (base) carregado!


In [ ]:
# 3. Sistema de Cache Inteligente

class CacheManager:
    def __init__(self, diretorio="cache"):
        self.diretorio = diretorio
        os.makedirs(diretorio, exist_ok=True)

    def _gerar_hash(self, texto):
        """Gera um hash único para o texto"""
        return hashlib.md5(texto.encode()).hexdigest()

    def salvar_resposta(self, pergunta, resposta):
        """Salva uma resposta no cache"""
        hash_pergunta = self._gerar_hash(pergunta)
        arquivo_cache = os.path.join(self.diretorio, f"{hash_pergunta}.json")

        dados = {
            "pergunta": pergunta,
            "resposta": resposta,
            "timestamp": datetime.now().isoformat()
        }

        with open(arquivo_cache, 'w', encoding='utf-8') as f:
            json.dump(dados, f, ensure_ascii=False, indent=2)

    def obter_resposta(self, pergunta):
        """Recupera uma resposta do cache se existir"""
        hash_pergunta = self._gerar_hash(pergunta)
        arquivo_cache = os.path.join(self.diretorio, f"{hash_pergunta}.json")

        if os.path.exists(arquivo_cache):
            with open(arquivo_cache, 'r', encoding='utf-8') as f:
                dados = json.load(f)
                print(f"📦 Resposta recuperada do cache!")
                return dados["resposta"]
        return None

cache = CacheManager()

In [ ]:
# 4. Histórico de Conversa

class HistoricoConversa:
    def __init__(self, max_mensagens=10):
        self.mensagens = [
            {"role": "system", "content": "Você é um assistente útil e responde em português de forma concisa e amigável."}
        ]
        self.max_mensagens = max_mensagens

    def adicionar(self, role, content):
        """Adiciona uma mensagem ao histórico"""
        self.mensagens.append({"role": role, "content": content})

        # Mantém apenas as últimas N mensagens (excluindo a system)
        if len(self.mensagens) > self.max_mensagens + 1:
            self.mensagens = [self.mensagens[0]] + self.mensagens[-(self.max_mensagens):]

    def obter_mensagens(self):
        """Retorna o histórico completo"""
        return self.mensagens

    def salvar(self, nome_arquivo="historico_conversa.json"):
        """Salva o histórico em arquivo"""
        caminho = os.path.join("historico", nome_arquivo)
        with open(caminho, 'w', encoding='utf-8') as f:
            json.dump(self.mensagens, f, ensure_ascii=False, indent=2)

    def carregar(self, nome_arquivo="historico_conversa.json"):
        """Carrega o histórico de arquivo"""
        caminho = os.path.join("historico", nome_arquivo)
        if os.path.exists(caminho):
            with open(caminho, 'r', encoding='utf-8') as f:
                self.mensagens = json.load(f)

historico = HistoricoConversa()

In [ ]:
# 5. Interface Gráfica Aprimorada com ipywidgets

# Cria a interface com widgets
botao_gravar = widgets.Button(
    description='🎤 Gravar',
    button_style='primary',
    layout=widgets.Layout(width='150px', height='50px')
)

botao_parar = widgets.Button(
    description='⏹️ Parar',
    button_style='danger',
    layout=widgets.Layout(width='150px', height='50px'),
    disabled=True
)

botao_processar = widgets.Button(
    description='⚙️ Processar',
    button_style='success',
    layout=widgets.Layout(width='150px', height='50px'),
    disabled=True
)

texto_pergunta = widgets.Textarea(
    value='',
    placeholder='Sua pergunta aparecerá aqui...',
    description='Pergunta:',
    disabled=True,
    layout=widgets.Layout(width='100%', height='100px')
)

texto_resposta = widgets.Textarea(
    value='',
    placeholder='Resposta do ChatGPT aparecerá aqui...',
    description='Resposta:',
    disabled=True,
    layout=widgets.Layout(width='100%', height='150px')
)

status_label = widgets.HTML(value='<b>Status:</b> Pronto')

# Layout da interface
interface = widgets.VBox([
    widgets.HTML("<h2>🎙️ Assistente de Voz Aprimorado</h2>"),
    widgets.HBox([botao_gravar, botao_parar, botao_processar]),
    status_label,
    texto_pergunta,
    texto_resposta
])

display(interface)


In [ ]:
# 6. Gravação Aprimorada com Detecção de Silêncio

class GravadorAvancado:
    def __init__(self):
        self.audio_buffer = []
        self.gravando = False
        self.audio_queue = queue.Queue()

    def _detectar_silencio(self, audio_data, threshold=500):
        """Detecta se o áudio contém silêncio"""
        return np.abs(audio_data).mean() < threshold

    def gravar_com_vad(self, tempo_maximo=10, silencio_maximo=1.5):
        """
        Grava áudio com detecção automática de silêncio (VAD)
        """
        import sounddevice as sd
        import numpy as np

        print("🎤 Gravando... (fale agora)")
        status_label.value = '<b>Status:</b> 🎤 Gravando...'

        self.gravando = True
        self.audio_buffer = []

        # Configurações de gravação
        samplerate = 16000
        duration = tempo_maximo
        frames_silencio = 0
        frames_silencio_max = int(silencio_maximo * samplerate / 1024)

        def callback(indata, frames, time, status):
            if self.gravando:
                self.audio_buffer.append(indata.copy())

        # Inicia gravação em thread separada
        stream = sd.InputStream(callback=callback, samplerate=samplerate, channels=1)
        stream.start()

        # Aguarda até detectar silêncio ou tempo máximo
        start_time = time.time()
        ultimo_som = time.time()

        while self.gravando and (time.time() - start_time) < tempo_maximo:
            time.sleep(0.1)

            # Verifica se há áudio recente para detectar silêncio
            if len(self.audio_buffer) > 0:
                ultimo_frame = self.audio_buffer[-1]
                if self._detectar_silencio(ultimo_frame, CONFIG["nivel_ruido"]):
                    if (time.time() - ultimo_som) > silencio_maximo:
                        self.gravando = False
                else:
                    ultimo_som = time.time()

        stream.stop()
        stream.close()

        # Concatena todos os buffers
        if len(self.audio_buffer) > 0:
            audio_array = np.concatenate(self.audio_buffer)

            # Salva como arquivo WAV
            from scipy.io import wavfile
            wavfile.write('gravacao.wav', samplerate, audio_array)

            status_label.value = '<b>Status:</b> ✅ Gravação concluída!'
            return 'gravacao.wav'
        else:
            status_label.value = '<b>Status:</b> ❌ Nenhum áudio gravado'
            return None

gravador = GravadorAvancado()

In [ ]:
# 7. Funções Aprimoradas de Processamento

def transcrever_com_whisper(caminho_audio):
    """Transcreve áudio com Whisper e tratamento de erros"""
    try:
        status_label.value = '<b>Status:</b> 🔄 Transcrevendo áudio...'
        resultado = modelo_whisper.transcribe(caminho_audio, language=CONFIG["idioma"])
        texto = resultado["text"].strip()

        if texto:
            status_label.value = '<b>Status:</b> ✅ Transcrição concluída!'
            return texto
        else:
            status_label.value = '<b>Status:</b> ⚠️ Nenhum texto detectado'
            return None

    except Exception as e:
        status_label.value = f'<b>Status:</b> ❌ Erro na transcrição: {str(e)}'
        return None

def perguntar_ao_chatgpt_com_historico(pergunta):
    """Envia pergunta ao ChatGPT com histórico e cache"""

    # Verifica cache primeiro
    if CONFIG["cache_habilitado"]:
        resposta_cache = cache.obter_resposta(pergunta)
        if resposta_cache:
            return resposta_cache

    try:
        status_label.value = '<b>Status:</b> 🤔 Consultando ChatGPT...'

        # Adiciona pergunta ao histórico
        historico.adicionar("user", pergunta)

        # Obtém resposta
        resposta = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=historico.obter_mensagens(),
            temperature=0.7,
            max_tokens=500
        )

        texto_resposta = resposta.choices[0].message.content

        # Adiciona resposta ao histórico
        historico.adicionar("assistant", texto_resposta)

        # Salva no cache
        if CONFIG["cache_habilitado"]:
            cache.salvar_resposta(pergunta, texto_resposta)

        status_label.value = '<b>Status:</b> ✅ Resposta recebida!'
        return texto_resposta

    except Exception as e:
        status_label.value = f'<b>Status:</b> ❌ Erro no ChatGPT: {str(e)}'
        return f"Desculpe, ocorreu um erro: {str(e)}"

def falar_com_gtts_aprimorado(texto):
    """Sintetiza voz com gTTS e melhorias"""
    try:
        status_label.value = '<b>Status:</b> 🔊 Sintetizando voz...'

        # Cria áudio com gTTS
        tts = gTTS(
            text=texto,
            lang=CONFIG["idioma"],
            slow=CONFIG["velocidade_gtts"]
        )

        # Salva em arquivo temporário
        arquivo_audio = 'resposta_temp.mp3'
        tts.save(arquivo_audio)

        # Carrega com pydub para melhorias
        audio = AudioSegment.from_mp3(arquivo_audio)

        # Aplica normalização (ajusta volume)
        audio = normalize(audio)

        # Exporta versão melhorada
        arquivo_final = 'resposta_melhorada.mp3'
        audio.export(arquivo_final, format='mp3')

        # Remove arquivos temporários
        os.remove(arquivo_audio)

        status_label.value = '<b>Status:</b> 🔊 Reproduzindo resposta...'

        # Retorna objeto de áudio para reprodução
        return ipd.Audio(arquivo_final, autoplay=True)

    except Exception as e:
        status_label.value = f'<b>Status:</b> ❌ Erro na síntese de voz: {str(e)}'
        return None

In [ ]:
# 8. Funções de Controle da Interface

def on_gravar_clicked(b):
    """Callback quando botão gravar é pressionado"""
    botao_gravar.disabled = True
    botao_parar.disabled = False
    botao_processar.disabled = True

    # Inicia gravação em thread separada para não travar a interface
    thread = threading.Thread(target=processar_gravacao)
    thread.start()

def on_parar_clicked(b):
    """Callback quando botão parar é pressionado"""
    gravador.gravando = False
    botao_gravar.disabled = False
    botao_parar.disabled = True

def on_processar_clicked(b):
    """Callback quando botão processar é pressionado"""
    pergunta = texto_pergunta.value

    if pergunta:
        # Processa em thread separada
        thread = threading.Thread(target=processar_pergunta, args=(pergunta,))
        thread.start()

def processar_gravacao():
    """Processa a gravação de áudio"""
    arquivo = gravador.gravar_com_vad(
        tempo_maximo=CONFIG["tempo_maximo_gravacao"],
        silencio_maximo=CONFIG["silencio_maximo"]
    )

    if arquivo:
        # Transcreve automaticamente
        pergunta = transcrever_com_whisper(arquivo)
        if pergunta:
            texto_pergunta.value = pergunta
            botao_processar.disabled = False

    botao_gravar.disabled = False
    botao_parar.disabled = True

def processar_pergunta(pergunta):
    """Processa a pergunta e gera resposta"""
    resposta = perguntar_ao_chatgpt_com_historico(pergunta)

    if resposta:
        texto_resposta.value = resposta
        audio = falar_com_gtts_aprimorado(resposta)
        if audio:
            display(audio)

    status_label.value = '<b>Status:</b> ✅ Pronto'

# Conecta os botões às funções
botao_gravar.on_click(on_gravar_clicked)
botao_parar.on_click(on_parar_clicked)
botao_processar.on_click(on_processar_clicked)

print("\n✅ Sistema pronto! Use os botões acima para interagir.")


✅ Sistema pronto! Use os botões acima para interagir.


In [ ]:
# 9. Funções Extras e Utilitários

def limpar_cache():
    """Limpa o cache de respostas"""
    import shutil
    if os.path.exists("cache"):
        shutil.rmtree("cache")
        os.makedirs("cache", exist_ok=True)
    print("🧹 Cache limpo!")

def exportar_historico():
    """Exporta o histórico de conversa"""
    historico.salvar()
    print("📝 Histórico exportado para 'historico_conversa.json'")

def alterar_modelo_whisper(modelo):
    """Altera o modelo do Whisper em tempo real"""
    global modelo_whisper
    modelos_validos = ['tiny', 'base', 'small', 'medium', 'large']

    if modelo in modelos_validos:
        status_label.value = f'<b>Status:</b> 🔄 Carregando modelo {modelo}...'
        modelo_whisper = whisper.load_model(modelo)
        CONFIG["modelo_whisper"] = modelo
        status_label.value = f'<b>Status:</b> ✅ Modelo {modelo} carregado!'
    else:
        print(f"❌ Modelo inválido. Opções: {modelos_validos}")

# Widget para seleção de modelo
seletor_modelo = widgets.Dropdown(
    options=['tiny', 'base', 'small', 'medium', 'large'],
    value='base',
    description='Modelo:',
    disabled=False
)

botao_carregar_modelo = widgets.Button(
    description='Carregar Modelo',
    button_style='info'
)

def on_carregar_modelo(b):
    alterar_modelo_whisper(seletor_modelo.value)

botao_carregar_modelo.on_click(on_carregar_modelo)

# Adiciona controles extras à interface
interface_extra = widgets.VBox([
    widgets.HTML("<h3>⚙️ Configurações Avançadas</h3>"),
    widgets.HBox([seletor_modelo, botao_carregar_modelo]),
    widgets.HBox([
        widgets.Button(description='🧹 Limpar Cache', on_click=lambda x: limpar_cache()),
        widgets.Button(description='📥 Exportar Histórico', on_click=lambda x: exportar_historico())
    ])
])

display(interface_extra)